In [2]:
import numpy as np

class BatchNorm1D:
    def __init__(self, dim, eps=1e-5, momentum=0.9):
        # learnable parameters
        self.gamma = np.ones((1, dim))
        self.beta  = np.zeros((1, dim))
        # running (inference) stats
        self.running_mean = np.zeros((1, dim))
        self.running_var  = np.ones((1, dim))
        # hyper‑params
        self.eps = eps
        self.momentum = momentum
        # cache for backward
        self.cache = None

    def forward(self, X, training=True):
        m, _ = X.shape

        if training:
            # 1) batch stats
            mu    = X.mean(axis=0, keepdims=True)
            var   = X.var(axis=0, keepdims=True)
            std   = np.sqrt(var + self.eps)
            X_hat = (X - mu) / std

            # update running stats
            self.running_mean = (
                self.momentum * self.running_mean
                + (1 - self.momentum) * mu
            )
            self.running_var = (
                self.momentum * self.running_var
                + (1 - self.momentum) * var
            )

            # save for backward
            self.cache = (X, X_hat, mu, var, std, m)
        else:
            # inference: use running stats
            std   = np.sqrt(self.running_var + self.eps)
            X_hat = (X - self.running_mean) / std
            # no cache needed

        # 2) scale & shift
        out = self.gamma * X_hat + self.beta
        return out

    def backward(self, d_out):
        """
        d_out: ∂L/∂Z_tilde, shape (m, dim)
        Returns:
          dX     : ∂L/∂X, same shape
          dγ, dβ : ∂L/∂γ, ∂L/∂β, shape (1, dim)
        """
        X, X_hat, mu, var, std, m = self.cache

        # 1) grad w.r.t. gamma & beta
        d_gamma = np.sum(d_out * X_hat, axis=0, keepdims=True)
        d_beta  = np.sum(d_out, axis=0, keepdims=True)

        # 2) grad w.r.t. X_hat
        d_xhat = d_out * self.gamma

        # 3) grad w.r.t. variance
        #    d_var = sum_i d_xhat_i * (X_i - μ) * (−½)(σ²+ε)^(-3/2)
        d_var = np.sum(
            d_xhat * (X - mu) * (-0.5) * std**(-3),
            axis=0,
            keepdims=True
        )

        # 4) grad w.r.t. mean
        #    d_mu   = sum_i d_xhat_i * (−1/σ) + d_var * (−2/m) sum(X_i-μ)
        d_mu = np.sum(
            d_xhat * (-1/std),
            axis=0,
            keepdims=True
        ) + d_var * np.mean(-2 * (X - mu), axis=0, keepdims=True)

        # 5) grad w.r.t. X
        #    dX = d_xhat / σ + d_var * 2*(X-μ)/m + d_mu / m
        dX = (
            d_xhat / std
            + d_var * 2 * (X - mu) / m
            + d_mu / m
        )

        return dX, d_gamma, d_beta

## 2. Walk‑through of the Backward Steps

1. **Scale & Shift Gradients**

   * $\displaystyle d\gamma = \sum_i \frac{\partial L}{\partial Z_{\text{norm},i}} \cdot \hat Z_i$
   * $\displaystyle d\beta = \sum_i \frac{\partial L}{\partial Z_{\text{norm},i}}$

2. **Normalized‑Value Gradient**

   * $\displaystyle d\hat{Z} = dZ_{\text{norm}} \cdot \gamma$

3. **Variance Gradient**

   * $\displaystyle d\sigma^2 = \sum_i d\hat{Z}_i \cdot (Z_i - \mu) \cdot \bigl(-\tfrac12\bigr) \cdot (\sigma^2+\epsilon)^{-\tfrac32}$

4. **Mean Gradient**

   * From both the normalization shift and how variance depends on μ:

   $$
   d\mu = \sum_i d\hat{Z}_i \cdot \bigl(-1/\sigma\bigr) \;+\; d\sigma^2 \cdot \frac{-2}{m}\sum_i(Z_i-\mu)
   $$

5. **Input Gradient**

   * Combine contributions:

   $$
   dX_i = \frac{d\hat{Z}_i}{\sigma} + d\sigma^2 \cdot \frac{2(Z_i-\mu)}{m} + \frac{d\mu}{m}
   $$

## 4. Putting It into a Full Network

When you integrate this into a training loop:

1. **Forward**

   * `Z = W·A_prev + b`
   * `Z_norm = batchnorm.forward(Z)`
   * `A = activation(Z_norm)`

2. **Backward**

   * `dZ_norm = dA * activation'(Z_norm)`
   * `dZ, dγ, dβ = batchnorm.backward(dZ_norm)`
   * `dW = dZ·A_prevᵀ`
   * `db = sum(dZ)`
   * `dA_prev = Wᵀ·dZ`

3. **Parameter Updates**

   * $W \leftarrow W - \eta \cdot dW$, etc.
   * $\gamma \leftarrow \gamma - \eta \cdot d\gamma$
   * $\beta \leftarrow \beta - \eta \cdot d\beta$

In [6]:
np.random.seed(0)
bn = BatchNorm1D(dim=1)
# fake batch of 4 scalars
X = np.array([[1.0], [2.0], [3.0], [4.0]])
# forward
Z_tilde = bn.forward(X, training=True)

# suppose upstream gradient is all ones
d_out = np.ones_like(Z_tilde)
# backward
dX, d_gamma, d_beta = bn.backward(d_out)

print("Z_tilde:\n", Z_tilde.flatten())
print("dX:\n", dX.flatten())
print("d_gamma:", d_gamma.flatten())
print("d_beta: ", d_beta.flatten())


Z_tilde:
 [-1.34163542 -0.44721181  0.44721181  1.34163542]
dX:
 [0. 0. 0. 0.]
d_gamma: [0.]
d_beta:  [4.]
